# [Notebook]Unihack26_Parcel3D_Lineage

Read source model NAME/OBJ from each render_config.yaml. Group normal/deformed instances by Google Scanned Objects source identity before proposing a split. This index supplies no cross-view correspondence or evidence-sufficiency labels. Original rendered images stay research-only; this notebook copies no images or textures.

In [ ]:
from pathlib import Path
import sys
SOURCES = {'ml/__init__.py': '', 'ml/data/__init__.py': '', 'ml/data/labels.py': '"""Validate pair supervision and detect split leakage before feature extraction."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nfrom collections import defaultdict\n\nSPLITS = ("train", "validation", "test")\nCORRESPONDENCE = {"MATCH", "WRONG_REGION", "WRONG_OBJECT", "UNKNOWN"}\nSUFFICIENCY = {"SUFFICIENT", "INSUFFICIENT", "UNKNOWN"}\nCHANGE = {"CHANGED", "UNCHANGED", "UNKNOWN"}\nSHA256 = re.compile(r"^[a-f0-9]{64}$")\nREASONS = {"BLUR", "GLARE", "OCCLUSION", "ROI_OUTSIDE_OVERLAP", "LOW_RESOLUTION",\n           "VIEW_MISMATCH", "NO_REFERENCE", "UNKNOWN"}\n\n\ndef locked_split(lineages: list[str], seed: str = "unihack26-v1") -> dict:\n    """Split a fixed, deduplicated corpus 60/20/20. Persist; never refit mid-run."""\n    if len(lineages) != len(set(lineages)) or any(not item for item in lineages):\n        raise ValueError("LINEAGE_IDS_MUST_BE_UNIQUE_AND_NONEMPTY")\n    if len(lineages) < 5:\n        raise ValueError("INSUFFICIENT_LINEAGES_FOR_THREE_SPLITS")\n    ordered = sorted(lineages, key=lambda item: hashlib.sha256(f"{seed}:{item}".encode()).digest())\n    train_end, validation_end = int(len(ordered) * 0.6), int(len(ordered) * 0.8)\n    groups = {name: split for split, names in zip(\n        SPLITS, (ordered[:train_end], ordered[train_end:validation_end], ordered[validation_end:]), strict=True)\n        for name in names}\n    payload = {"version": "lineage-split-v1", "seed": seed, "lineages": groups}\n    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)\n    return {**payload, "sha256": hashlib.sha256(canonical.encode()).hexdigest()}\n\n\ndef valid_polygons(polygons) -> bool:\n    if not isinstance(polygons, list):\n        return False\n    for polygon in polygons:\n        if not isinstance(polygon, list) or len(polygon) < 3:\n            return False\n        for point in polygon:\n            if not isinstance(point, list) or len(point) != 2:\n                return False\n            if any(isinstance(v, bool) or not isinstance(v, (int, float))\n                   or not math.isfinite(v) or not 0 <= v <= 1 for v in point):\n                return False\n        area = abs(sum(polygon[i][0] * polygon[(i + 1) % len(polygon)][1]\n                       - polygon[(i + 1) % len(polygon)][0] * polygon[i][1]\n                       for i in range(len(polygon)))) / 2\n        if area <= 1e-8:\n            return False\n    return True\n\n\ndef validate_pairs(records: list[dict], split_manifest: dict) -> dict:\n    """Return explicit failures. UNKNOWN is masked, never interpreted as a negative."""\n    errors = []\n    seen_pairs = set()\n    hashes = defaultdict(set)\n    queries = defaultdict(set)\n    mapping = split_manifest.get("lineages", {})\n    counts = defaultdict(int)\n    for index, record in enumerate(records):\n        def fail(code):\n            errors.append({"row": index, "pair_id": record.get("pair_id"), "code": code})\n\n        required = ("pair_id", "query_id", "split", "query_lineage_id", "reference_lineage_id",\n                    "query_sha256", "reference_sha256", "source_kind", "source_ref", "license_ref",\n                    "annotator_id", "correspondence", "sufficiency", "observable_change",\n                    "change_polygons", "ignore_polygons", "reason_codes")\n        if any(key not in record for key in required):\n            fail("MISSING_REQUIRED_FIELDS")\n            continue\n        text_fields = set(required) - {"change_polygons", "ignore_polygons", "reason_codes"}\n        if any(not isinstance(record[key], str) or not record[key].strip() for key in text_fields):\n            fail("INVALID_TEXT_FIELD")\n            continue\n        pair_id, split = record["pair_id"], record["split"]\n        if pair_id in seen_pairs:\n            fail("DUPLICATE_PAIR_ID")\n        seen_pairs.add(pair_id)\n        if split not in SPLITS:\n            fail("INVALID_SPLIT")\n        counts[split] += 1\n        for side in ("query", "reference"):\n            if mapping.get(record[f"{side}_lineage_id"]) != split:\n                fail(f"{side.upper()}_LINEAGE_SPLIT_MISMATCH")\n            digest = record[f"{side}_sha256"]\n            if not SHA256.fullmatch(digest):\n                fail("INVALID_IMAGE_SHA256")\n            hashes[digest].add(split)\n        queries[record["query_id"]].add(split)\n        correspondence, sufficiency, change = (record[key] for key in (\n            "correspondence", "sufficiency", "observable_change"))\n        if correspondence not in CORRESPONDENCE or sufficiency not in SUFFICIENCY or change not in CHANGE:\n            fail("INVALID_LABEL_ENUM")\n        if correspondence != "MATCH" and sufficiency == "SUFFICIENT":\n            fail("SUFFICIENT_REQUIRES_MATCH")\n        if change != "UNKNOWN" and (correspondence != "MATCH" or sufficiency != "SUFFICIENT"):\n            fail("CHANGE_REQUIRES_SUFFICIENT_MATCH")\n        if correspondence in ("MATCH", "WRONG_REGION") and record["query_lineage_id"] != record["reference_lineage_id"]:\n            fail("SAME_OBJECT_LABEL_LINEAGE_MISMATCH")\n        if correspondence == "WRONG_OBJECT" and record["query_lineage_id"] == record["reference_lineage_id"]:\n            fail("WRONG_OBJECT_REQUIRES_DIFFERENT_LINEAGE")\n        if record["source_kind"] not in {"REAL", "SYNTHETIC", "SOFTWARE_FIXTURE"}:\n            fail("INVALID_SOURCE_KIND")\n        if not valid_polygons(record["change_polygons"]) or not valid_polygons(record["ignore_polygons"]):\n            fail("INVALID_NORMALIZED_POLYGON")\n        if record["change_polygons"] and change != "CHANGED":\n            fail("CHANGE_MASK_WITHOUT_CHANGED_LABEL")\n        if not isinstance(record["reason_codes"], list) or any(not isinstance(reason, str) or reason not in REASONS for reason in record["reason_codes"]):\n            fail("INVALID_REASON_CODES")\n    for digest, splits in hashes.items():\n        if len(splits) > 1:\n            errors.append({"code": "IMAGE_HASH_SPLIT_LEAKAGE", "sha256": digest, "splits": sorted(splits)})\n    for query_id, splits in queries.items():\n        if len(splits) > 1:\n            errors.append({"code": "QUERY_SPLIT_LEAKAGE", "query_id": query_id, "splits": sorted(splits)})\n    return {"gate": "PASS" if records and not errors else "FAIL", "rows": len(records),\n            "split_counts": dict(counts), "errors": errors,\n            "license_clearance": "SEPARATE_REVIEW_REQUIRED", "near_duplicate_audit": "SEPARATE_REVIEW_REQUIRED"}\n\n\ndef supervision(record: dict) -> dict:\n    match = record["correspondence"]\n    sufficient = record["sufficiency"]\n    return {\n        "same_region_target": float(match == "MATCH"),\n        "same_region_mask": match != "UNKNOWN",\n        "sufficiency_target": float(sufficient == "SUFFICIENT"),\n        "sufficiency_mask": match == "MATCH" and sufficient != "UNKNOWN",\n        "change_mask": match == "MATCH" and sufficient == "SUFFICIENT" and record["observable_change"] != "UNKNOWN",\n    }\n', 'ml/data/parcel3d_audit.py': '"""CPU-only audit of a mounted Parcel3D input. Does not copy its image archive."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nfrom collections import Counter, deque\nfrom datetime import UTC, datetime\nfrom pathlib import Path, PurePosixPath\n\nfrom PIL import Image\n\nSOURCE = "ayesha19765/parcel3d"\nORIGINAL = "https://zenodo.org/records/8032204"\nALLOWED_SPLITS = ("train", "validation", "test")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef within_root(root: Path, relative: str) -> Path:\n    relative = relative.replace("\\\\", "/")\n    parsed = PurePosixPath(relative)\n    if parsed.is_absolute() or ".." in parsed.parts or ":" in relative:\n        raise ValueError("UNSAFE_ANNOTATION_PATH")\n    result = (root / Path(*parsed.parts)).resolve()\n    if not result.is_relative_to(root.resolve()):\n        raise ValueError("PATH_OUTSIDE_INPUT")\n    return result\n\n\ndef find_annotations(input_root: Path) -> dict[str, Path]:\n    # Kaggle can expose both legacy and owner-qualified aliases of one input.\n    # Annotation files sit above per-image directories; never walk 13,400 samples\n    # just to locate three files. Bound discovery and compare duplicate contents.\n    roots, queue, visited = [], deque([(input_root, 0)]), set()\n    while queue:\n        directory, depth = queue.popleft()\n        resolved = directory.resolve()\n        if resolved in visited:\n            continue\n        visited.add(resolved)\n        if len(visited) > 200:\n            raise RuntimeError("ANNOTATION_DISCOVERY_LIMIT")\n        if all((directory / f"parcel3d_{split}.json").is_file() for split in ALLOWED_SPLITS):\n            roots.append(directory)\n            continue\n        if depth < 5:\n            for child in sorted(directory.iterdir()):\n                if (\n                    child.name not in {*ALLOWED_SPLITS, "misc"}\n                    and not child.name.startswith(".")\n                    and child.is_dir()\n                ):\n                    queue.append((child, depth + 1))\n    if not roots:\n        raise RuntimeError("ANNOTATION_ROOT_NOT_FOUND")\n    roots.sort(key=lambda path: (len(path.parts), str(path)))\n    if len(roots) > 1:\n        digests = {\n            tuple(sha256_file(root / f"parcel3d_{split}.json") for split in ALLOWED_SPLITS)\n            for root in roots\n        }\n        if len(digests) != 1:\n            raise RuntimeError("ANNOTATION_ALIASES_HAVE_DIFFERENT_CONTENT")\n    print(f"Annotation roots: {len(roots)} equivalent mount(s); selected {roots[0]}", flush=True)\n    return {split: roots[0] / f"parcel3d_{split}.json" for split in ALLOWED_SPLITS}\n\n\ndef resolve_image(annotation_path: Path, image_record: dict, split: str) -> Path | None:\n    filename = image_record.get("file_name")\n    if not isinstance(filename, str):\n        return None\n    candidates = [\n        within_root(annotation_path.parent, filename),\n        within_root(annotation_path.parent, f"{split}/{filename}"),\n    ]\n    for candidate in candidates:\n        if candidate.is_file():\n            return candidate\n    return None\n\n\ndef audit(input_root: Path, output_root: Path, source_version: int = 1) -> dict:\n    input_root = input_root.resolve()\n    output_root = output_root.resolve()\n    if output_root.is_relative_to(input_root):\n        raise ValueError("OUTPUT_MUST_NOT_BE_INSIDE_READ_ONLY_INPUT")\n    output_root.mkdir(parents=True, exist_ok=True)\n    print("Locating Parcel3D annotation roots (bounded discovery)", flush=True)\n    annotations = find_annotations(input_root)\n    result = {\n        "schema_version": "parcel3d-audit-v2",\n        "source": SOURCE,\n        "source_version": source_version,\n        "original_record": ORIGINAL,\n        "checked_at": datetime.now(UTC).isoformat(),\n        "runtime": {"python": platform.python_version(), "platform": platform.platform()},\n        "source_kind": "SYNTHETIC",\n        "splits": {},\n        "errors": [],\n        "license": {\n            "mirror_metadata": "Apache-2.0; not authoritative for upstream assets",\n            "original_full_dataset": "Academic research only; Zenodo other-nc",\n            "textureless_obj_meshes": "CC-BY-4.0, separate grant in original record",\n            "cardboard_textures": "CC-BY-4.0, separate record 8041823",\n            "training_policy": "Full rendered-image training remains gated pending scope review.",\n            "commercial_release": "NOT_CLEARED_FOR_ORIGINAL_RENDERED_IMAGES",\n            "reference_matching_labels": "Must inspect; COCO damage labels alone do not establish cross-view correspondence or evidence sufficiency.",\n        },\n    }\n    for package in ("numpy", "Pillow", "torch", "opencv-python"):\n        try:\n            result["runtime"][package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    group_sets = {}\n    sample_directories = set()\n    for split, path in annotations.items():\n        print(f"Auditing {split}: {path.name}", flush=True)\n        document = json.loads(path.read_text(encoding="utf-8"))\n        images = document.get("images", [])\n        labels = document.get("annotations", [])\n        record = {\n            "annotation_path": str(path.relative_to(input_root)),\n            "annotation_sha256": sha256_file(path),\n            "annotation_bytes": path.stat().st_size,\n            "top_level_keys": sorted(document),\n            "image_count": len(images),\n            "annotation_count": len(labels),\n            "categories": document.get("categories", []),\n            "image_fields": dict(Counter(k for image in images for k in image)),\n            "annotation_fields": dict(Counter(k for annotation in labels for k in annotation)),\n            "sample_records": images[:2],\n            "sample_annotations": labels[:2],\n            "samples": [],\n        }\n        ids = [image.get("id") for image in images]\n        record["duplicate_image_ids"] = len(ids) - len(set(ids))\n        group_key = next(\n            (\n                key\n                for key in ("mesh_id", "object_id", "parcel_id", "model_id")\n                if images and all(key in item for item in images)\n            ),\n            None,\n        )\n        record["explicit_group_key"] = group_key\n        record["group_split_verified"] = False\n        if group_key:\n            group_sets[split] = {str(item[group_key]) for item in images}\n        # Deterministic stratified positions; no exhaustive image read or copy.\n        positions = sorted({int(i * max(len(images) - 1, 0) / 7) for i in range(8)})\n        for index in positions if images else []:\n            image_record = images[index]\n            image_path = resolve_image(path, image_record, split)\n            sample = {\n                "image_id": image_record.get("id"),\n                "file_name": image_record.get("file_name"),\n            }\n            if image_path is None:\n                sample["status"] = "PATH_UNRESOLVED"\n                result["errors"].append(\n                    {"split": split, "image_id": sample["image_id"], "code": "PATH_UNRESOLVED"}\n                )\n            else:\n                try:\n                    with Image.open(image_path) as image:\n                        image.load()\n                        sample.update(\n                            status="READ_OK",\n                            width=image.width,\n                            height=image.height,\n                            format=image.format,\n                            mode=image.mode,\n                        )\n                    sample["sha256"] = sha256_file(image_path)\n                    sample["bytes"] = image_path.stat().st_size\n                    sample["relative_path"] = str(image_path.relative_to(input_root))\n                    sample_directories.add(image_path.parent)\n                except (OSError, ValueError) as exc:\n                    sample.update(status="READ_FAILED", error_type=type(exc).__name__)\n                    result["errors"].append(\n                        {"split": split, "image_id": sample["image_id"], "code": "READ_FAILED"}\n                    )\n            record["samples"].append(sample)\n        result["splits"][split] = record\n    root = annotations["train"].parent\n    extensions = Counter()\n    mesh_examples = []\n    metadata_examples = []\n    # Inventory only sampled image neighborhoods. Do not imply a complete count.\n    print(f"Inspecting metadata/meshes near {len(sample_directories)} sampled images", flush=True)\n    for sample_directory in sorted(sample_directories):\n        for directory, _, filenames in os.walk(sample_directory):\n            for filename in sorted(filenames):\n                suffix = Path(filename).suffix.lower()\n                extensions[suffix] += 1\n                path = Path(directory) / filename\n                if suffix == ".obj" and len(mesh_examples) < 8:\n                    mesh_examples.append(\n                        {\n                            "path": str(path.relative_to(root)),\n                            "bytes": path.stat().st_size,\n                            "sha256": sha256_file(path),\n                        }\n                    )\n                if (\n                    suffix == ".json"\n                    and path not in annotations.values()\n                    and len(metadata_examples) < 8\n                ):\n                    meta = {"path": str(path.relative_to(root)), "bytes": path.stat().st_size}\n                    if path.stat().st_size < 100_000:\n                        try:\n                            meta["document"] = json.loads(path.read_text(encoding="utf-8"))\n                        except (ValueError, UnicodeError):\n                            meta["status"] = "UNREADABLE_JSON"\n                    metadata_examples.append(meta)\n    result["inventory"] = {\n        "scope": "SAMPLED_IMAGE_NEIGHBORHOODS",\n        "sample_directory_count": len(sample_directories),\n        "sampled_extension_counts": dict(extensions),\n        "mesh_examples": mesh_examples,\n        "metadata_examples": metadata_examples,\n    }\n    if len(group_sets) == 3:\n        overlaps = {\n            f"{a}:{b}": len(group_sets[a] & group_sets[b])\n            for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))\n        }\n        result["group_overlap_counts"] = overlaps\n        for record in result["splits"].values():\n            record["group_split_verified"] = not any(overlaps.values())\n    result["mount_read_gate"] = (\n        "PASS"\n        if not result["errors"]\n        and all(\n            any(s["status"] == "READ_OK" for s in record["samples"])\n            for record in result["splits"].values()\n        )\n        else "FAIL"\n    )\n    result["correspondence_label_gate"] = "PENDING_REVIEW"\n    result["commercial_license_gate"] = "HOLD_ORIGINAL_RENDERS"\n    result["full_archive_copied"] = False\n    target = output_root / "parcel3d-audit.json"\n    target.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    (output_root / "audit-sha256.txt").write_text(\n        sha256_file(target) + "  parcel3d-audit.json\\n", encoding="ascii"\n    )\n    print(\n        json.dumps(\n            {\n                "mount_read_gate": result["mount_read_gate"],\n                "counts": {k: v["image_count"] for k, v in result["splits"].items()},\n                "outputs": str(output_root),\n            },\n            ensure_ascii=False,\n        )\n    )\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", type=Path, default=Path("/kaggle/input"))\n    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/unihack26-audit"))\n    parser.add_argument("--source-version", type=int, default=1)\n    args = parser.parse_args()\n    audit(args.input, args.output, args.source_version)\n', 'ml/data/parcel3d_lineage.py': '"""Recover source-object lineage from Parcel3D metadata without copying images."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path, PurePosixPath\n\nimport yaml\n\nfrom ml.data.labels import locked_split\nfrom ml.data.parcel3d_audit import find_annotations, sha256_file, within_root\n\n\ndef parse_lineage(config: dict, scenario_id: str, split: str, category_id: int) -> dict:\n    scenario = config.get("SCENARIO", {})\n    model = scenario.get("MODEL", {})\n    if scenario.get("ID") != scenario_id or scenario.get("FLAG") != split:\n        raise ValueError("SCENARIO_ID_OR_SPLIT_MISMATCH")\n    if scenario.get("CATEGORY_ID") != category_id:\n        raise ValueError("CATEGORY_MISMATCH")\n    name, obj = model.get("NAME"), model.get("OBJ")\n    if not isinstance(name, str) or not isinstance(obj, str):\n        raise ValueError("SOURCE_OBJECT_ID_MISSING")\n    parts = PurePosixPath(obj.replace("\\\\", "/"))\n    if parts.is_absolute() or ".." in parts.parts or ":" in obj:\n        raise ValueError("UNSAFE_SOURCE_OBJECT_PATH")\n    if len(parts.parts) < 4 or parts.parts[0] != "google_scanned_objects" or parts.parts[1] != name:\n        raise ValueError("UNRECOGNIZED_SOURCE_OBJECT_LINEAGE")\n    return {"lineage_id": f"google_scanned_objects/{name}", "source_mesh": obj,\n            "parcel_type": scenario.get("PARCEL_TYPE"),\n            "views_configured": config.get("RENDER", {}).get("VIEWS"),\n            "render_seed": config.get("RENDER", {}).get("SEED")}\n\n\ndef index_lineages(input_root: Path, output_root: Path, workers: int = 8) -> dict:\n    annotations = find_annotations(input_root)\n    root = annotations["train"].parent\n    output_root.mkdir(parents=True, exist_ok=True)\n    requests, errors = [], []\n    annotation_hashes = {}\n    for split, path in annotations.items():\n        document = json.loads(path.read_text())\n        annotation_hashes[split] = sha256_file(path)\n        by_image = defaultdict(list)\n        for annotation in document["annotations"]:\n            by_image[annotation["image_id"]].append(annotation)\n        for image in document["images"]:\n            associated = by_image[image["id"]]\n            if len(associated) != 1:\n                errors.append({"split": split, "image_id": image["id"], "code": "EXPECTED_ONE_PARCEL"})\n                continue\n            requests.append((split, image, associated[0]))\n\n    def read_one(item):\n        split, image, annotation = item\n        try:\n            image_path = within_root(root, image["file_name"])\n            scenario_dir = image_path.parent\n            metadata_path = scenario_dir / "meta/render_config.yaml"\n            if metadata_path.stat().st_size > 64_000:\n                raise ValueError("METADATA_SIZE_LIMIT")\n            raw = metadata_path.read_bytes()\n            config = yaml.safe_load(raw)\n            info = parse_lineage(config, scenario_dir.name, split, annotation["category_id"])\n            mesh_path = within_root(root, annotation["model"])\n            if mesh_path.parent.parent != scenario_dir:\n                raise ValueError("MESH_SCENARIO_MISMATCH")\n            return {"source_split": split, "image_id": image["id"], "scenario_id": scenario_dir.name,\n                    "image_path": image["file_name"], "mesh_path": annotation["model"],\n                    "category_id": annotation["category_id"], "metadata_sha256": hashlib.sha256(raw).hexdigest(), **info}\n        except (OSError, ValueError, KeyError, TypeError, AttributeError, yaml.YAMLError) as exc:\n            known = str(exc) if isinstance(exc, ValueError) else "METADATA_UNREADABLE"\n            return {"error": {"split": split, "image_id": image["id"], "code": known}}\n\n    records = []\n    with ThreadPoolExecutor(max_workers=max(1, min(workers, 16))) as pool:\n        for index, result in enumerate(pool.map(read_one, requests)):\n            if "error" in result:\n                errors.append(result["error"])\n            else:\n                records.append(result)\n            if (index + 1) % 500 == 0:\n                print(f"Lineage metadata {index + 1}/{len(requests)}", flush=True)\n    groups = defaultdict(set)\n    scenarios = Counter()\n    for record in records:\n        groups[record["lineage_id"]].add(record["source_split"])\n        scenarios[record["scenario_id"]] += 1\n    duplicates = [identifier for identifier, count in scenarios.items() if count != 1]\n    overlaps = {lineage: sorted(splits) for lineage, splits in groups.items() if len(splits) > 1}\n    result = {"version": "parcel3d-lineage-v1", "source": "ayesha19765/parcel3d", "source_version": 1,\n              "annotation_hashes": annotation_hashes, "indexed_records": len(records),\n              "source_object_lineages": len(groups), "source_split_counts": dict(Counter(r["source_split"] for r in records)),\n              "source_lineages_crossing_splits": overlaps, "duplicate_scenario_ids": duplicates,\n              "metadata_gate": "PASS" if records and not errors and not duplicates else "FAIL",\n              "original_source_object_split_gate": "PASS" if records and not errors and not overlaps and not duplicates else "FAIL",\n              "correspondence_labels": "NOT_PROVIDED_BY_THIS_INDEX", "errors": errors,\n              "full_image_hash_audit": "NOT_RUN", "commercial_render_license": "HOLD_ORIGINAL_RENDERS"}\n    if len(groups) >= 5 and not errors and not duplicates:\n        manifest = locked_split(sorted(groups))\n        for record in records:\n            record["group_split"] = manifest["lineages"][record["lineage_id"]]\n        (output_root / "lineage-split.json").write_text(json.dumps(manifest, indent=2))\n        result["proposed_split_sha256"] = manifest["sha256"]\n        result["proposed_split_lineage_counts"] = dict(Counter(manifest["lineages"].values()))\n        result["proposed_split_record_counts"] = dict(Counter(r["group_split"] for r in records))\n    target = output_root / "lineage-index.jsonl"\n    target.write_text("".join(json.dumps(record, ensure_ascii=False) + "\\n" for record in records), encoding="utf-8")\n    result["index_sha256"] = sha256_file(target)\n    (output_root / "lineage-audit.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(json.dumps({key: value for key, value in result.items() if key not in {"errors", "source_lineages_crossing_splits"}}))\n    return result\n'}
bundle = Path('/kaggle/working/_src')
for name, source in SOURCES.items():
    path = bundle / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(bundle))


In [ ]:
from ml.data.parcel3d_lineage import index_lineages
result = index_lineages(Path('/kaggle/input'), Path('/kaggle/working/lineage'), workers=8)
assert result['metadata_gate'] == 'PASS', result['errors'][:5]
